# ResNet-20 Precision Lab\nCompare three trained weight configurations on PYNQ-Z2. Use the live tab for one image, then open the hardware tab for measured layer cycles and memory accesses. The comparison tab reads measured results from the board benchmark.

In [ ]:
import io
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
from PIL import Image

from precision_benchmark import (COLORS, accuracy, exact_count, latency_stats,
                                 load_suite, read_counters, select_configuration)
from pynq_full_model import CLASSES, FullModel

data_dir = Path("benchmark_data")
suite = load_suite(data_dir)
sheet = Image.open(data_dir / "cifar10_preview.png")
results_path = Path("benchmark_results.json")
precision = widgets.Dropdown(options=[("All INT8", "int8"), ("All INT4", "int4"),
                                      ("Mixed", "mixed")], value="mixed", description="Weights")
index = widgets.IntSlider(min=0, max=len(suite["mixed"][1]) - 1, value=0,
                          description="Image index", continuous_update=False)
run = widgets.Button(description="Run on FPGA", button_style="primary", icon="play")
image = widgets.Image(width=280, height=280)
caption = widgets.HTML()
live, compare, hardware = (widgets.Output() for _ in range(3))
floor = widgets.FloatSlider(min=0, max=1, step=.01, value=.85,
                            description="Accuracy floor", continuous_update=False)
session = {}


def selected():
    model, cases = suite[precision.value]
    return model, cases[index.value]


def preview(*_):
    _, case = selected()
    i = int(case["name"].split("[")[1].rstrip("]"))
    x, y = (i % 10) * 32, (i // 10) * 32
    buffer = io.BytesIO()
    sheet.crop((x, y, x + 32, y + 32)).resize((280, 280), getattr(Image, "Resampling", Image).NEAREST).save(buffer, "PNG")
    image.value = buffer.getvalue()
    caption.value = f"<b>{case['name']}</b> &nbsp; Ground truth: {CLASSES[case['label']]}"
    live.clear_output(wait=False)
    hardware.clear_output(wait=False)
    with hardware:
        print("Run the selected image to see measured FPGA execution.")


def execution(model, cycles, memory, clock):
    names, values = zip(*cycles)
    colors = [COLORS[{1: "bnn", 4: "int4", 8: "int8"}[model["precision"][name]]]
              if name in model["precision"] else "#8D99A6" for name in names]
    diagram, flow = plt.subplots(figsize=(10, 1.15), constrained_layout=True)
    nodes = ("RGB image", "Conv1", "Stage 1", "Stage 2", "Stage 3", "Avg pool", "10 scores")
    for i, node in enumerate(nodes):
        flow.text(i, 0, node, va="center", ha="center", color="white",
                  bbox=dict(boxstyle="round,pad=.45", facecolor="#203A53", edgecolor="none"))
        if i:
            flow.annotate("", xy=(i - .30, 0), xytext=(i - .66, 0),
                          arrowprops=dict(arrowstyle="->", color="#60798E"))
    flow.set(xlim=(-.65, len(nodes) - .35), ylim=(-.5, .5))
    flow.set_axis_off()
    plt.show()
    plt.close(diagram)
    print(f"{clock:g} MHz | {sum(values):,} cycles")
    for title, value in memory.items():
        print(f"{title.replace('_', ' ').capitalize()}: {value:,}")
    fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
    ax.barh(names, values, color=colors)
    ax.invert_yaxis()
    ax.set_xlabel("FPGA cycles")
    ax.set_title("Measured cycles by operation")
    keys = (("INT8", COLORS["int8"]), ("INT4", COLORS["int4"]), ("Binary", COLORS["bnn"]),
            ("Add / projection / pool", "#8D99A6"))
    ax.legend(handles=[Patch(color=color, label=label) for label, color in keys],
              frameon=False, loc="lower right")
    plt.show()
    plt.close(fig)


@live.capture(clear_output=True, wait=True)
def infer(_):
    run.disabled = True
    try:
        model, case = selected()
        if session.get("variant") != precision.value:
            engine = FullModel("programmable.bit", model, profile=True)
            session.update(variant=precision.value, engine=engine)
            print(f"Model upload: {engine.model_load_ms:.1f} ms (once)")
        engine = session["engine"]
        result = engine.infer(case["input"])
        matches = exact_count(case, result)
        print(f"Prediction: {CLASSES[result['prediction']]} | {engine.clock_mhz:g} MHz")
        print(f"{matches}/10 exact CPU matches, {10 - matches} differing")
        print(f"FPGA {result['compute_ms']:.2f} ms | Upload through scores {result['total_ms']:.2f} ms")
        fig, ax = plt.subplots(figsize=(6.4, 3))
        x = range(len(CLASSES))
        ax.bar([i - .18 for i in x], case["scores"], .36, color="#8895A7", label="CPU")
        ax.bar([i + .18 for i in x], result["scores"], .36, color=COLORS[precision.value], label="FPGA")
        ax.set_xticks(list(x))
        ax.set_xticklabels(CLASSES, rotation=30, ha="right")
        ax.set_ylabel("Logit")
        ax.legend(frameon=False)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        stages, counters = read_counters(engine, model)
        with hardware:
            hardware.clear_output(wait=True)
            execution(model, stages, counters, engine.clock_mhz)
    finally:
        run.disabled = False


@compare.capture(clear_output=True, wait=True)
def comparison(*_):
    if not results_path.exists():
        print("CPU reference results. Run the board benchmark to measure FPGA accuracy and latency.")
        print(f"{'Variant':<8} {'CPU accuracy':>13} {'Packed weights':>16}")
        for variant, (model, cases) in suite.items():
            correct = sum(max(range(10), key=case["scores"].__getitem__) == case["label"] for case in cases)
            print(f"{variant:<8} {correct:>6}/{len(cases):<6} {len(model['weights']) * 4 / 1024:>12.1f} KiB")
        return
    groups = json.loads(results_path.read_text())
    print(f"{'Variant':<7} {'Policy':<9} {'N':>4} {'Acc':>7} {'Exact':>8} {'Weights KiB':>12} "
          f"{'Mean ms':>10} {'Median':>9} {'P95':>9} {'Min':>9} {'Max':>9}")
    for g in groups:
        r = g["rows"]
        t = latency_stats([x["policy_ms"] for x in r])
        print(f"{g['variant']:<7} {g['policy']:<9} {t['n']:>4} {accuracy(r):>7.1%} "
              f"{sum(x['exact_outputs'] == 10 for x in r):>4}/{t['n']:<3} {g['weight_bytes']/1024:>12.1f} "
              f"{t['mean']:>10.2f} {t['median']:>9.2f} {t['p95']:>9.2f} "
              f"{t['minimum']:>9.2f} {t['maximum']:>9.2f}")
        print(f"  Model upload {g['upload_ms']:.1f} ms once | Parameters {g['parameter_bytes']:,} B "
              f"| Weight bytes/image {g['per_image_weight_bytes']:,} B | "
              f"FPGA {r[0]['cycles']:,} cycles / {r[0]['compute_ms']:.2f} ms")
    fig, ax = plt.subplots(figsize=(8.5, 4), constrained_layout=True)
    for g in groups:
        r = g["rows"]
        ax.scatter(latency_stats([v["policy_ms"] for v in r])["median"], accuracy(r) * 100,
                   c=COLORS[g["variant"]], marker="o" if g["policy"] == "resident" else "s",
                   label=f"{g['variant']} / {g['policy']}", s=75)
    ax.set(xlabel="Median per-image policy latency (ms)", ylabel="Accuracy (%)",
           title="Accuracy and latency across precision and weight policies")
    ax.legend(frameon=False, ncol=2)
    plt.show()
    plt.close(fig)
    resident = [g for g in groups if g["policy"] == "resident"]
    if resident:
        fig, (size_ax, phase_ax) = plt.subplots(1, 2, figsize=(10.5, 3.7), constrained_layout=True)
        variants = [g["variant"] for g in resident]
        sizes = [g["weight_bytes"] / 1024 for g in resident]
        sizes_bars = size_ax.barh(variants, sizes, color=[COLORS[v] for v in variants])
        for bar, size in zip(sizes_bars, sizes):
            size_ax.text(size + 2, bar.get_y() + bar.get_height() / 2, f"{size:.1f}", va="center")
        size_ax.set(xlabel="Packed weights (KiB)", title="Model storage", xlim=(0, max(sizes) * 1.22))
        left = [0.0] * len(variants)
        for title, key, color in (("Image upload", "image_load_ms", "#7891AA"),
                                  ("FPGA compute", "compute_ms", "#1C8B77"),
                                  ("START / polling", "control_ms", "#B6A06E"),
                                  ("Score read and scale", "readback_ms", "#DA805E")):
            means = [latency_stats([r[key] + (r["score_scale_ms"] if key == "readback_ms" else 0)
                                   for r in g["rows"]])["mean"] for g in resident]
            phase_ax.barh(variants, means, left=left, color=color, label=title)
            left = [a + b for a, b in zip(left, means)]
        phase_ax.set(xlabel="Mean time (ms)", title="Resident inference breakdown")
        phase_ax.legend(frameon=False, fontsize=8, loc="lower right")
        plt.show()
        plt.close(fig)
    try:
        for row in select_configuration(groups, floor.value):
            print(f"{row['policy']} / {row['method']}: {row['variant']} | "
                  f"holdout n={row['n']}, accuracy {row['accuracy']:.1%}, "
                  f"predicted {row['predicted_ms']:.1f} ms, measured {row['measured_ms']:.1f} ms")
    except ValueError as error:
        print(error)


precision.observe(preview, names="value")
index.observe(preview, names="value")
floor.observe(comparison, names="value")
run.on_click(infer)
refresh = widgets.Button(description="Reload measurements", icon="refresh")
refresh.on_click(comparison)
tabs = widgets.Tab(children=[
    widgets.VBox([widgets.HBox([precision, index, run]), caption, widgets.HBox([image, live])]),
    widgets.VBox([widgets.HBox([refresh, floor]), compare]),
    hardware])
for i, title in enumerate(("Live inference", "Precision comparison", "Hardware execution")):
    tabs.set_title(i, title)
preview()
comparison()
display(widgets.HTML("""
<div style="background:#172A40;color:#F4F7FB;padding:16px 20px;border-radius:8px;margin-bottom:10px">
<div style="font-size:23px;font-weight:700">ResNet-20 Precision Lab</div>
<div style="color:#B8C9DC;margin-top:4px">PYNQ-Z2 | CIFAR-10 | 32 operations per inference</div>
</div>
"""), tabs)
